# ClearRoute - YOLO11 Fine-Tuning
**Goal:** train YOLO11n on a street litter dataset so `detect.py` can recognise real waste instead of generic COCO objects.

Run each cell in order from top to bottom. The whole notebook takes ~20 minutes on a free T4 GPU.

## Cell 1 - Check GPU
YOLO training is much faster on a GPU. This cell confirms that Colab gave you one.

If it says **No GPU found**, go to **Runtime > Change runtime type > T4 GPU > Save** and run again.

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)

if result.returncode == 0:
    for line in result.stdout.splitlines():
        if any(g in line for g in ['Tesla', 'T4', 'A100', 'L4', 'V100']):
            print('GPU found:', line.strip())
            break
    else:
        print('GPU is active. Full output:')
        print(result.stdout)
else:
    print('No GPU found!')
    print('Go to Runtime > Change runtime type > T4 GPU > Save, then run again.')


## Cell 2 - Install Dependencies
`ultralytics` is the YOLO11 library. `roboflow` lets us download the dataset with one line of code.

The `!` at the start of a line means: run this command in the terminal, not in Python.

In [ ]:
!pip install ultralytics roboflow --quiet

import ultralytics, roboflow
print(f'ultralytics {ultralytics.__version__}')
print(f'roboflow    {roboflow.__version__}')


## Cell 3 - Download the Trash AI v2 Dataset
Trash AI v2 is a public dataset on Roboflow Universe with annotated images of street litter.

**Before running this cell you need to:**
1. Create a free account at roboflow.com
2. Go to account settings > API Keys > copy your **Private API Key**
3. Paste it below replacing `YOUR_API_KEY_HERE`


In [ ]:
from roboflow import Roboflow

# REPLACE THIS VALUE WITH YOUR ROBOFLOW API KEY
API_KEY = 'YOUR_API_KEY_HERE'

if API_KEY == 'YOUR_API_KEY_HERE':
    raise ValueError(
        'Paste your Roboflow API key before running this cell.\n'
        'Go to roboflow.com > Settings > API Keys and copy your key.'
    )

rf = Roboflow(api_key=API_KEY)

# Trash AI v2 dataset on Roboflow Universe.
project = rf.workspace('trash-ai-27vay').project('trash-ai-v2')
dataset = project.version(1).download('yolov11')

print('Dataset downloaded to:', dataset.location)


## Cell 4 - Verify the Dataset
Before training, check that images and annotations downloaded correctly.
You should see image counts for train / val / test and 4 sample images.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

dataset_root = Path(dataset.location)

for split in ['train', 'valid', 'test']:
    img_dir = dataset_root / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*.*')))
        print(f'  {split:6s}: {count} images')
    else:
        print(f'  {split:6s}: folder not found')

train_images = (
    list((dataset_root / 'train' / 'images').glob('*.jpg')) +
    list((dataset_root / 'train' / 'images').glob('*.png'))
)

if not train_images:
    print('No training images found - check the folder structure above.')
else:
    samples = random.sample(train_images, min(4, len(train_images)))
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('4 random training images', fontsize=14)
    for ax, img_path in zip(axes, samples):
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('Dataset looks good - ready to train.')


## Cell 5 - Train the Model
We start from `yolo11n.pt` (trained on 80 generic objects) and continue training on the litter dataset.
This is called **fine-tuning** - the model keeps prior knowledge and learns new litter classes on top.

| Parameter | Value | Why |
|-----------|-------|-----|
| `epochs` | 30 | Good for a hackathon; increase to 100 for higher accuracy later |
| `imgsz` | 640 | Standard YOLO input size |
| `batch` | 16 | Images per step - reduce to 8 if you see out-of-memory errors |
| `patience` | 10 | Stops early if accuracy does not improve for 10 epochs |

Training takes roughly **10-20 minutes** on a T4 GPU.

In [ ]:
from ultralytics import YOLO

# Start from the nano model - lightest and fastest, good for real-time use
model = YOLO('yolo11n.pt')

# data.yaml was created automatically by the Roboflow download (Cell 3)
data_yaml = str(dataset_root / 'data.yaml')
print('Training config:', data_yaml)

results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project='clearroute_training',
    name='trash_ai_v1',
    exist_ok=True,  # overwrite if you re-run this cell
)

print('Training complete!')


## Cell 6 - Evaluate the Result
We run the trained model on the **validation set** (images it never saw during training).

**Key metrics:**
- **mAP50** - main score (0-1). Above 0.5 is decent; above 0.7 is good for a hackathon.
- **Precision** - of everything flagged as litter, how often was it right?
- **Recall** - of all the real litter in the images, how much did it find?


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

best_model_path = Path('clearroute_training/trash_ai_v1/weights/best.pt')

if not best_model_path.exists():
    print('best.pt not found - did the training cell finish without errors?')
else:
    trained_model = YOLO(str(best_model_path))
    metrics = trained_model.val(data=data_yaml, verbose=False)

    print('=== Validation Metrics ===')
    print(f'  mAP50     : {metrics.box.map50:.3f}')
    print(f'  Precision : {metrics.box.mp:.3f}')
    print(f'  Recall    : {metrics.box.mr:.3f}')

    # Show training curves (loss and accuracy over epochs)
    curves_path = Path('clearroute_training/trash_ai_v1/results.png')
    if curves_path.exists():
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(mpimg.imread(curves_path))
        ax.axis('off')
        ax.set_title('Training curves', fontsize=13)
        plt.tight_layout()
        plt.show()

    # Run inference on 4 validation images and show bounding boxes
    val_images = list((dataset_root / 'valid' / 'images').glob('*.jpg'))[:4]
    if val_images:
        preds = trained_model.predict(val_images, verbose=False)
        fig, axes = plt.subplots(1, len(preds), figsize=(20, 5))
        if len(preds) == 1:
            axes = [axes]
        fig.suptitle('Detections on validation images', fontsize=14)
        for ax, r in zip(axes, preds):
            ax.imshow(r.plot()[..., ::-1])  # YOLO returns BGR, flip to RGB
            ax.axis('off')
        plt.tight_layout()
        plt.show()


## Cell 7 - Export and Download the Model
Downloads `best.pt` to your computer.
This file replaces `yolo11n.pt` in `detect.py` - it is the only change needed to use the fine-tuned model.

In [ ]:
from google.colab import files
from pathlib import Path

best_model_path = Path('clearroute_training/trash_ai_v1/weights/best.pt')

if not best_model_path.exists():
    print('best.pt not found - make sure the training cell finished successfully.')
else:
    size_mb = best_model_path.stat().st_size / (1024 * 1024)
    print(f'Model size: {size_mb:.1f} MB')
    files.download(str(best_model_path))
    print('Download started. Save best.pt inside your clearroute/ folder.')
    print('Then open detect.py and change one line:')
    print('  Before: model = YOLO(yolo11n.pt)')
    print('  After : model = YOLO(best.pt)')


---
## What to do next

You now have `best.pt` - a YOLO11 model fine-tuned specifically on street litter.

**Steps to plug it into the ClearRoute pipeline:**

1. Move `best.pt` into your `clearroute/` folder (same folder as `detect.py`).
2. Open `detect.py` and change one line inside the model loading section:
   - Before: `model = YOLO('yolo11n.pt')`
   - After:  `model = YOLO('best.pt')`
3. Check `data.yaml` (inside the downloaded dataset folder) for the exact class names and update `LITTER_CLASSES` in `detect.py` to match.
4. Run `detect.py` as before - it now writes `dados_reais.json` with litter-specific detections.
5. Point `app.py` at `dados_reais.json` by changing one line inside `load_data()`.

Everything else - heatmap, priority queue, detail panel - works with no other changes.